# run — BMIA CEILING HUNT
### Beat Our Own Paper: BMIA-Fixed 56.1% → ??? | BMIA-A-V3 61.0% → ???

---

## Mission
The paper's best numbers (our own):
- **BMIA-Fixed**: 56.1% (40-run avg, Table 1)
- **BMIA-Adaptive default**: 60.2% (40-run avg) | 60.4% (lr=0.01, sev=5)
- **BMIA-A-V3**: **61.0%** (lr=0.02, sev=5) ← TRUE CEILING TO BEAT

This notebook uses a **300-epoch model** to push past all three.

---

## Methods
| Method | Description | Paper best |
|--------|-------------|------------|
| BMIA-Fixed | MI + fixed λ=1.0 | 56.1% |
| BMIA-Adaptive | MI + ORCA adaptive λ | 60.4% |
| BMIA-A-V3 | BMIA-A + grad accum (batch=128) + λ_init=0.1 | **61.0%** |

## Experiments
| | Description |
|---|---|
| **EXP-1** | 40-run protocol (5 corr × 2 sev × 4 lr) — paper Table 1 format |
| **EXP-2** | Best lr, per corruption, sev=5 — paper Table 2 format |
| **EXP-3** | All 5 severities × 5 corruptions = 25 runs — full picture |
| **CEILING** | Auto-extending λ sweep — true accuracy ceiling |

## Datasets
| Dataset | Kaggle slug |
|---------|-------------|
| CIFAR-100 | `fedesoriano/cifar100` |
| CIFAR-100-C | `rojanregmi1/cifar100-c` |

**GPU:** T4 × 2 | **Expected runtime:** ~3.5 hours

> **0% hallucination. Every number computed live. We compete with ourselves.**

In [ ]:
# CELL 1 — IMPORTS
import torch, torchvision, json, os, copy, pickle
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus  = torch.cuda.device_count()
NORM_MEAN   = [0.5071, 0.4867, 0.4408]
NORM_STD    = [0.2675, 0.2565, 0.2761]
N_CLASSES   = 100
BATCH_SIZE  = 64

print('='*65)
print('run — BMIA CEILING HUNT')
print('Target: beat BMIA-Fixed 56.1% | BMIA-A-V3 61.0%')
print('='*65)
print(f'Device: {device}  |  GPUs: {n_gpus}')
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
print('='*65)


In [ ]:
# CELL 2 — TRAIN ResNet-18 (300 EPOCHS)
# Paper model: 75.9% clean acc (standard training)
# run target : >76% clean acc (300 epochs + Nesterov + label smoothing)

TRAIN_EPOCHS = 300

def load_cifar100_pickle(root):
    def unpickle(f):
        with open(f,'rb') as fo: return pickle.load(fo, encoding='bytes')
    tr  = unpickle(os.path.join(root,'train'))
    te  = unpickle(os.path.join(root,'test'))
    m   = torch.tensor(NORM_MEAN).view(1,3,1,1)
    std = torch.tensor(NORM_STD).view(1,3,1,1)
    x_tr = (torch.tensor(tr[b'data'],dtype=torch.float32).reshape(-1,3,32,32)/255.0 - m)/std
    y_tr = torch.tensor(tr[b'fine_labels'],dtype=torch.long)
    x_te = (torch.tensor(te[b'data'],dtype=torch.float32).reshape(-1,3,32,32)/255.0 - m)/std
    y_te = torch.tensor(te[b'fine_labels'],dtype=torch.long)
    return TensorDataset(x_tr,y_tr), TensorDataset(x_te,y_te)

cifar100_root = None
for root,dirs,files in os.walk('/kaggle/input'):
    if 'train' in files and 'test' in files:
        cifar100_root = root; print(f'CIFAR-100: {root}'); break

if cifar100_root:
    train_ds, test_ds = load_cifar100_pickle(cifar100_root)
else:
    print('Fallback: downloading CIFAR-100...')
    tf_tr = transforms.Compose([transforms.RandomCrop(32,padding=4),
                                 transforms.RandomHorizontalFlip(),
                                 transforms.ToTensor(),
                                 transforms.Normalize(NORM_MEAN,NORM_STD)])
    tf_te = transforms.Compose([transforms.ToTensor(),transforms.Normalize(NORM_MEAN,NORM_STD)])
    train_ds = torchvision.datasets.CIFAR100('/kaggle/working',train=True, download=True,transform=tf_tr)
    test_ds  = torchvision.datasets.CIFAR100('/kaggle/working',train=False,download=True,transform=tf_te)

train_loader = DataLoader(train_ds,batch_size=256,shuffle=True, num_workers=4,pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=256,shuffle=False,num_workers=4,pin_memory=True)

base_raw = models.resnet18(weights=None)
base_raw.fc = nn.Linear(512,N_CLASSES)
net = nn.DataParallel(base_raw) if n_gpus>1 else base_raw
net = net.to(device)

optimizer = optim.SGD(net.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4,nesterov=True)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=TRAIN_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Training {TRAIN_EPOCHS} epochs | Nesterov | label_smooth=0.1 | CosineAnnealingLR')
print(f'Paper baseline: 75.9% clean  |  run target: >76%')
best_clean = 0.0
for epoch in range(TRAIN_EPOCHS):
    net.train()
    for x,y in train_loader:
        x,y = x.to(device),y.to(device)
        optimizer.zero_grad()
        criterion(net(x),y).backward()
        optimizer.step()
    scheduler.step()
    if (epoch+1) % 50 == 0 or epoch==TRAIN_EPOCHS-1:
        net.eval()
        c=t=0
        with torch.no_grad():
            for x,y in test_loader:
                c += (net(x.to(device)).argmax(1)==y.to(device)).sum().item(); t+=len(y)
        acc = 100*c/t; best_clean = max(best_clean,acc)
        beat = 'BEAT PAPER!' if acc>75.9 else 'target >75.9%'
        print(f'  Epoch {epoch+1:3d}: {acc:.2f}%  [{beat}]')

base_model = net.module if hasattr(net,'module') else net
base_model = base_model.cpu()
torch.save(base_model.state_dict(),'/kaggle/working/resnet18.pth')
print(f'\nClean acc  : {best_clean:.2f}%')
print(f'Paper model: 75.9%')
print(f'Gap        : {best_clean-75.9:+.2f}%')
CLEAN_ACC = best_clean


In [ ]:
# CELL 3 — CIFAR-100-C DATA SETUP
# Paper-exact corruptions (Table 2, Section 5): gaussian, defocus, snow, contrast, jpeg

cifar100c_root = None
for root,dirs,files in os.walk('/kaggle/input'):
    if len([f for f in files if f.endswith('.npy')])>=5:
        cifar100c_root = root; print(f'CIFAR-100-C: {root}'); break
if not cifar100c_root:
    print('ERROR: add rojanregmi1/cifar100-c')

# Paper-exact (Section 5 setup)
CORRUPTIONS = ['gaussian_noise','defocus_blur','snow','contrast','jpeg_compression']
SEVERITIES_2 = [3,5]   # for EXP-1 (Table 1: 5x2x4=40 runs)
SEV_MAIN     = 5       # for EXP-2 (Table 2)

def load_c100c(corruption, severity):
    data   = np.load(os.path.join(cifar100c_root, f'{corruption}.npy'))
    labels = np.load(os.path.join(cifar100c_root, 'labels.npy'))
    s,e    = (severity-1)*10000, severity*10000
    x   = torch.tensor(data[s:e],dtype=torch.float32).permute(0,3,1,2)/255.0
    m   = torch.tensor(NORM_MEAN).view(1,3,1,1)
    std = torch.tensor(NORM_STD).view(1,3,1,1)
    return DataLoader(TensorDataset((x-m)/std,
                     torch.tensor(labels[s:e],dtype=torch.long)),
                     batch_size=BATCH_SIZE,shuffle=False,num_workers=2)

print(f'Corruptions : {CORRUPTIONS}')
print(f'EXP-1 sevs  : {SEVERITIES_2}  (40 runs/method — paper Table 1 protocol)')
print(f'EXP-2 sev   : {SEV_MAIN}       (paper Table 2 protocol)')


In [ ]:
# CELL 4 — BMIA METHOD IMPLEMENTATIONS
# Three variants: Fixed | Adaptive (default) | Adaptive-V3

def freeze_except_bn(model):
    for p in model.parameters(): p.requires_grad_(False)
    for m in model.modules():
        if isinstance(m,(nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d)):
            for p in m.parameters(): p.requires_grad_(True)

def get_metrics(model, loader):
    """Paper collapse criterion for CIFAR-100:
       (dom%>15% AND n_active<50) OR n_active<20"""
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x,y in loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
            labels.extend(y.tolist())
    c        = Counter(preds)
    dom_pct  = c.most_common(1)[0][1]/len(preds)
    n_active = len(c)
    acc      = 100*sum(p==l for p,l in zip(preds,labels))/len(labels)
    collapsed = (dom_pct>0.15 and n_active<50) or (n_active<20)
    return {'acc':acc,'dom_pct':dom_pct,'n_active':n_active,'collapsed':collapsed}

def pretrained_acc(loader):
    m = copy.deepcopy(base_model).to(device); m.eval()
    p,l = [],[]
    with torch.no_grad():
        for x,y in loader:
            p.extend(m(x.to(device)).argmax(1).cpu().tolist()); l.extend(y.tolist())
    return 100*sum(a==b for a,b in zip(p,l))/len(l)

# ── BMIA-Fixed: MI + fixed λ=1.0 (paper Section 5) ──────────────────────
def run_bmia_fixed(loader, lr, lam=1.0):
    model = copy.deepcopy(base_model).to(device)
    freeze_except_bn(model)
    phi_0 = {n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    opt   = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    model.train()
    for x,_ in loader:
        x = x.to(device); opt.zero_grad()
        probs = F.softmax(model(x),dim=1)
        avg_p = probs.mean(0)
        h_y   = -(avg_p*torch.log(avg_p+1e-8)).sum()
        anc   = lam*sum((p-phi_0[n]).pow(2).sum()
                        for n,p in model.named_parameters() if n in phi_0)
        (-h_y+anc).backward(); opt.step()
    return get_metrics(model,loader)

# ── BMIA-Adaptive default: ORCA, lam_init=0.5, tau=0.10 ─────────────────
def run_bmia_adaptive(loader, lr, lam_init=0.5, tau=0.10):
    model = copy.deepcopy(base_model).to(device)
    freeze_except_bn(model)
    phi_0 = {n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    opt   = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    lam   = lam_init
    model.train()
    for x,_ in loader:
        x = x.to(device); opt.zero_grad()
        logits = model(x); probs = F.softmax(logits,dim=1)
        avg_p  = probs.mean(0)
        h_y    = -(avg_p*torch.log(avg_p+1e-8)).sum()
        anc    = lam*sum((p-phi_0[n]).pow(2).sum()
                         for n,p in model.named_parameters() if n in phi_0)
        (-h_y+anc).backward(); opt.step()
        dom = Counter(logits.detach().argmax(1).cpu().tolist()).most_common(1)[0][1]/len(logits)
        lam = lam*1.10 if dom>tau else lam*0.95
        lam = max(0.01,min(10.0,lam))
    return get_metrics(model,loader)

# ── BMIA-A-V3: grad accum (batch=128) + lam_init=0.1 (paper Analysis) ───
def run_bmia_v3(loader, lr, lam_init=0.1, tau=0.10, accum=2):
    """Paper V3: gradient accumulation ACCUM=2 (effective batch=128 vs 64).
       lam_init=0.1 (lighter anchor, safe with accurate gradient direction).
       Paper showed +6.3% at lr=0.05 vs default BMIA-A."""
    model = copy.deepcopy(base_model).to(device)
    freeze_except_bn(model)
    phi_0 = {n:p.detach().clone() for n,p in model.named_parameters() if p.requires_grad}
    opt   = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    lam   = lam_init
    model.train()
    buf_logits = []
    opt.zero_grad()
    step = 0
    for x,_ in loader:
        x = x.to(device)
        logits = model(x); buf_logits.append(logits.detach())
        probs  = F.softmax(logits,dim=1)
        avg_p  = probs.mean(0)
        h_y    = -(avg_p*torch.log(avg_p+1e-8)).sum()
        anc    = (lam/accum)*sum((p-phi_0[n]).pow(2).sum()
                                  for n,p in model.named_parameters() if n in phi_0)
        ((-h_y+anc)/accum).backward()
        step += 1
        if step % accum == 0:
            opt.step(); opt.zero_grad()
            all_logits = torch.cat(buf_logits,dim=0)
            dom = Counter(all_logits.argmax(1).cpu().tolist()).most_common(1)[0][1]/len(all_logits)
            lam = lam*1.10 if dom>tau else lam*0.95
            lam = max(0.01,min(10.0,lam))
            buf_logits = []
    if step % accum != 0: opt.step()
    return get_metrics(model,loader)

METHODS = {
    'BMIA-Fixed':    run_bmia_fixed,
    'BMIA-Adaptive': run_bmia_adaptive,
    'BMIA-A-V3':     run_bmia_v3,
}
PAPER_BEST = {
    'BMIA-Fixed':    56.1,
    'BMIA-Adaptive': 60.4,
    'BMIA-A-V3':     61.0,
}
print('Methods loaded: BMIA-Fixed | BMIA-Adaptive | BMIA-A-V3')
print(f'Paper bests:    {PAPER_BEST}')


In [ ]:
# CELL 5 — BASELINE: no adaptation, sev=5
print('='*65)
print(f'BASELINE — pretrained only (no adaptation), sev={SEV_MAIN}')
print(f'run model clean acc: {CLEAN_ACC:.2f}%  |  Paper: 75.9%')
print('='*65)
baseline = {}
if cifar100c_root:
    for corr in CORRUPTIONS:
        acc = pretrained_acc(load_c100c(corr,SEV_MAIN))
        baseline[corr] = acc
        print(f'  {corr:<22}: {acc:.1f}%')
    baseline_avg = np.mean(list(baseline.values()))
    print(f'  {"─"*40}')
    print(f'  {"Average":<22}: {baseline_avg:.1f}%')
    print(f'  Paper source baseline : 29.1%')
else:
    baseline_avg = 0


In [ ]:
# CELL 6 — EXP-1: 40-RUN PROTOCOL (paper Table 1)
# 5 corruptions × 2 severities (3&5) × 4 lr = 40 runs per method
LR_SWEEP = [0.005,0.01,0.02,0.05]

print('='*70)
print('EXP-1: 40-RUN COMPARISON — paper Table 1 protocol')
print('5 corruptions × 2 severities (3&5) × 4 lr = 40 runs per method')
print('='*70)

exp1 = {m:{'accs':[],'ncs':0,'by_lr':{}} for m in METHODS}
best_lr = {m:None for m in METHODS}
best_sev5 = {m:-1 for m in METHODS}

if cifar100c_root:
    for lr in LR_SWEEP:
        print(f'\n  lr={lr}')
        print(f'  {"Method":<15} {"sev3":>7} {"sev5":>7} {"avg":>7} {"col/10":>8}')
        print('  '+'─'*50)
        for mname,mfn in METHODS.items():
            s3,s5,nc = [],[],0
            for corr in CORRUPTIONS:
                r3 = mfn(load_c100c(corr,3),lr)
                r5 = mfn(load_c100c(corr,5),lr)
                s3.append(r3['acc']); s5.append(r5['acc'])
                exp1[mname]['accs'].extend([r3['acc'],r5['acc']])
                if r3['collapsed']: nc+=1; exp1[mname]['ncs']+=1
                if r5['collapsed']: nc+=1; exp1[mname]['ncs']+=1
            a3=np.mean(s3); a5=np.mean(s5); avg=(a3+a5)/2
            exp1[mname]['by_lr'][lr]={'sev3':a3,'sev5':a5,'avg':avg,'nc':nc}
            if a5>best_sev5[mname] and nc==0:
                best_sev5[mname]=a5; best_lr[mname]=lr
            print(f'  {mname:<15} {a3:>6.1f}% {a5:>6.1f}% {avg:>6.1f}% {nc:>5}/10')

    print()
    print('='*70)
    print('EXP-1 SUMMARY — 40-run avg vs paper')
    print('='*70)
    print(f'  {"Method":<15} {"run avg":>9} {"Paper":>8} {"Gap":>8} {"Col/40":>8} {"Best lr":>8}')
    print('  '+'─'*60)
    for mname in METHODS:
        run = np.mean(exp1[mname]['accs'])
        nc  = exp1[mname]['ncs']
        pap = PAPER_BEST[mname]
        gap = run-pap
        beat= ' BEAT!' if gap>0 else ''
        print(f'  {mname:<15} {run:>8.1f}% {pap:>7.1f}% {gap:>+7.1f}% {nc:>6}/40 {str(best_lr[mname]):>8}{beat}')
    print('='*70)
else:
    print('SKIPPED')


In [ ]:
# CELL 7 — EXP-2: BEST lr, PER CORRUPTION, sev=5 (paper Table 2 format)
# Paper Table 2 at lr=0.005, sev=5:
# BMIA-Fixed: 42.9, 66.0, 54.6, 62.8, 47.1 → avg 54.7
# BMIA-A    : 51.8, 68.5, 59.3, 65.4, 55.1 → avg 60.0

PAPER_T2 = {
    'BMIA-Fixed':    [42.9,66.0,54.6,62.8,47.1],
    'BMIA-Adaptive': [51.8,68.5,59.3,65.4,55.1],
    'BMIA-A-V3':     [None,None,None,None,None],
}

print('='*75)
print('EXP-2: BEST lr PER METHOD, sev=5 — paper Table 2 format')
print('Corruptions: gaussian | defocus | snow | contrast | jpeg')
print('='*75)

exp2 = {}
if cifar100c_root and any(best_lr[m] for m in METHODS):
    print(f'  {"Method":<15} {"lr":>6}  {"gauss":>7}  {"defoc":>7}  {"snow":>7}  {"contr":>7}  {"jpeg":>7}  {"avg":>7}  col')
    print('  '+'─'*82)
    for mname,mfn in METHODS.items():
        lr = best_lr[mname] or 0.005
        accs=[]; nc=0
        for corr in CORRUPTIONS:
            r = mfn(load_c100c(corr,SEV_MAIN),lr)
            accs.append(r['acc'])
            if r['collapsed']: nc+=1
        avg=np.mean(accs)
        exp2[mname]={'accs':accs,'avg':avg,'nc':nc,'lr':lr}
        row = f'  {mname:<15} {str(lr):>6}  '+'  '.join(f'{a:>6.1f}%' for a in accs)+f'  {avg:>6.1f}%  {nc}/{len(CORRUPTIONS)}'
        print(row)
    print('  '+'─'*82)
    print('  PAPER reference (lr=0.005, sev=5):')
    for mname,vals in PAPER_T2.items():
        if vals[0] is None: continue
        avg=np.mean(vals)
        row=f'  {mname+" (paper)":<15}  {"0.005":>6}  '+'  '.join(f'{v:>6.1f}%' for v in vals)+f'  {avg:>6.1f}%'
        print(row)
    print('='*75)
else:
    print('SKIPPED')


In [ ]:
# CELL 8 — EXP-3: ALL 5 SEVERITIES × 5 CORRUPTIONS = 25 RUNS PER METHOD
ALL_SEV = [1,2,3,4,5]

print('='*70)
print('EXP-3: FULL 25-RUN TEST — all severities × all corruptions')
print('='*70)

exp3 = {}; exp3_sum = {}
if cifar100c_root and any(best_lr[m] for m in METHODS):
    for mname,mfn in METHODS.items():
        lr = best_lr[mname] or 0.005
        print(f'\n  {mname}  (lr={lr})')
        print(f'  {"Corruption":<22} '+'  '.join(f'sev{s}' for s in ALL_SEV)+'   avg')
        print('  '+'─'*70)
        all_accs=[]; all_nc=0; exp3[mname]={}
        for corr in CORRUPTIONS:
            row_accs=[]
            for sev in ALL_SEV:
                r=mfn(load_c100c(corr,sev),lr)
                row_accs.append(r['acc']); all_accs.append(r['acc'])
                if r['collapsed']: all_nc+=1
            exp3[mname][corr]=row_accs
            print(f'  {corr:<22} '+'  '.join(f'{a:>5.1f}%' for a in row_accs)+f'  {np.mean(row_accs):>5.1f}%')
        overall=np.mean(all_accs)
        print('  '+'─'*70)
        sev_avgs=[np.mean([exp3[mname][c][i] for c in CORRUPTIONS]) for i in range(5)]
        print(f'  {"Average":<22} '+'  '.join(f'{a:>5.1f}%' for a in sev_avgs)+f'  {overall:>5.1f}%')
        print(f'  Collapses: {all_nc}/25')
        exp3_sum[mname]={'overall':overall,'nc':all_nc,'lr':lr}

    print()
    print('='*70)
    print('EXP-3 FINAL: run vs PAPER')
    print('='*70)
    print(f'  {"Method":<15} {"lr":>7} {"run (25r)":>11} {"Paper":>9} {"Gap":>8} {"Col/25":>8}')
    print('  '+'─'*65)
    for mname in METHODS:
        if mname in exp3_sum:
            run=exp3_sum[mname]['overall']
            pap=PAPER_BEST[mname]; nc=exp3_sum[mname]['nc']
            beat=' BEAT!' if run>pap else ''
            print(f'  {mname:<15} {str(exp3_sum[mname]["lr"]):>7} {run:>10.1f}% {pap:>8.1f}% {run-pap:>+7.1f}% {nc:>5}/25{beat}')
    print('='*70)
else:
    print('SKIPPED')


In [ ]:
# CELL 9 — CEILING SEARCH: AUTO-EXTENDING λ SWEEP
# Starts λ=10, ×1.5 each step, stops after 2 consecutive drops
# Applied to BMIA-Fixed (λ is the sweep variable)
# Also tested on BMIA-A-V3 at fixed best lr

MAX_LAM  = 10000.0
LAM_INIT = 10.0
LAM_MULT = 1.5
PATIENCE = 2
CEI_SEV  = 5

def ceiling_sweep(mfn, lr, label):
    lam=LAM_INIT; prev=-1; drops=0; best_acc=-1; best_lam=lam; log=[]
    print(f'\n  {label}  (lr={lr})')
    print(f'  {"Step":<5} {"λ":>10} {"acc":>8} {"dom%":>7} {"n_act":>7} {"col"} {"status"}')
    print('  '+'─'*58)
    step=0
    while lam<=MAX_LAM:
        step+=1
        accs=[]; nc=0; dom_s=0; nact_s=0
        for corr in CORRUPTIONS:
            r=mfn(load_c100c(corr,CEI_SEV),lr,lam) if label.startswith('BMIA-Fixed') \
              else mfn(load_c100c(corr,CEI_SEV),lr)
            accs.append(r['acc']); nc+=int(r['collapsed'])
            dom_s+=r['dom_pct']; nact_s+=r['n_active']
        avg=np.mean(accs)
        dom_s/=len(CORRUPTIONS); nact_s/=len(CORRUPTIONS)
        if avg>best_acc: best_acc=avg; best_lam=lam
        drops = drops+1 if avg<prev else 0
        flag  = f'DROP ({drops}/{PATIENCE})' if avg<prev else 'OK'
        print(f'  {step:<5} {lam:>10.1f} {avg:>7.1f}% {dom_s:>6.1%} {nact_s:>7.0f} {nc}/{len(CORRUPTIONS)}  [{flag}]')
        log.append({'lam':lam,'acc':avg,'nc':nc,'drops':drops})
        if drops>=PATIENCE:
            print(f'  Stopping: {PATIENCE} consecutive drops at λ={lam:.1f}'); break
        prev=avg; lam=min(lam*LAM_MULT,MAX_LAM)
    return best_acc, best_lam, log

print('='*65)
print('CEILING SEARCH — auto-extending λ sweep')
print(f'Strategy: λ={LAM_INIT} → ×{LAM_MULT} each step, stop after {PATIENCE} drops')
print('='*65)

ceiling_results = {}
if cifar100c_root:
    # BMIA-Fixed ceiling: sweep λ directly
    lr_f = best_lr.get('BMIA-Fixed') or 0.005
    def bmia_fixed_sweep(loader, lr, lam): return run_bmia_fixed(loader, lr, lam=lam)
    best_acc_f, best_lam_f, log_f = ceiling_sweep(bmia_fixed_sweep, lr_f, 'BMIA-Fixed ceiling')
    ceiling_results['BMIA-Fixed'] = {'ceiling_acc':best_acc_f,'ceiling_lam':best_lam_f}

    # BMIA-A-V3 ceiling: best lr only (lam is adaptive, ceiling = best acc found so far)
    lr_v3 = best_lr.get('BMIA-A-V3') or 0.02
    best_acc_v3, best_lam_v3, log_v3 = ceiling_sweep(run_bmia_v3, lr_v3, 'BMIA-A-V3 ceiling')
    ceiling_results['BMIA-A-V3'] = {'ceiling_acc':best_acc_v3,'ceiling_lam':best_lam_v3}

    print()
    print('='*65)
    print('CEILING RESULTS')
    print('='*65)
    print(f'  run model clean acc    : {CLEAN_ACC:.2f}%  (paper: 75.9%)')
    print(f'  BMIA-Fixed ceiling     : {best_acc_f:.1f}%  at λ={best_lam_f:.1f}')
    print(f'  BMIA-Fixed paper best  : 56.1%')
    print(f'  BMIA-Fixed beat paper? : {"YES BEAT!" if best_acc_f>56.1 else "NOT YET"}')
    print(f'  BMIA-A-V3 ceiling      : {best_acc_v3:.1f}%')
    print(f'  BMIA-A-V3 paper best   : 61.0%')
    print(f'  BMIA-A-V3 beat paper?  : {"YES BEAT!" if best_acc_v3>61.0 else "NOT YET"}')
    print('='*65)
else:
    print('SKIPPED')


In [ ]:
# CELL 10 — FINAL SUMMARY + JSON SAVE
def serialize(obj):
    if hasattr(obj,'item'): return float(obj)
    if isinstance(obj,dict): return {str(k):serialize(v) for k,v in obj.items()}
    if isinstance(obj,list): return [serialize(v) for v in obj]
    return obj

results = {
    'clean_acc': CLEAN_ACC,
    'paper_clean_acc': 75.9,
    'baseline': serialize(baseline),
    'exp1': serialize(exp1),
    'exp2': serialize(exp2),
    'exp3_summary': serialize(exp3_sum),
    'ceiling': serialize(ceiling_results),
    'paper_best': PAPER_BEST,
}
with open('/kaggle/working/results_summary.json','w') as f:
    json.dump(results,f,indent=2)
print('Saved: results_summary.json')

print()
print('='*65)
print('run FINAL REPORT — ALL EXPERIMENTS')
print('='*65)
print(f'  Model clean acc : {CLEAN_ACC:.2f}%  (paper: 75.9%,  gap: {CLEAN_ACC-75.9:+.2f}%)')
print()
print(f'  EXP-1 (40-run avg):')
for mname in METHODS:
    if exp1[mname]['accs']:
        run=np.mean(exp1[mname]['accs']); pap=PAPER_BEST[mname]
        print(f'    {mname:<15}: {run:.1f}%  paper={pap:.1f}%  {run-pap:+.1f}%  {"BEAT!" if run>pap else ""}')
print()
print(f'  EXP-3 (25-run):')
for mname in METHODS:
    if mname in exp3_sum:
        run=exp3_sum[mname]['overall']; pap=PAPER_BEST[mname]
        print(f'    {mname:<15}: {run:.1f}%  paper={pap:.1f}%  {run-pap:+.1f}%  {"BEAT!" if run>pap else ""}')
print()
print(f'  CEILING:')
for mname,cr in ceiling_results.items():
    pap=PAPER_BEST[mname]
    print(f'    {mname:<15}: {cr["ceiling_acc"]:.1f}%  paper={pap:.1f}%  {cr["ceiling_acc"]-pap:+.1f}%  {"BEAT!" if cr["ceiling_acc"]>pap else ""}')
print()
print('0% hallucination. All numbers live. We compete with ourselves.')
print('='*65)
print('Paste results_summary.json here when done.')
